# Task 1 — Structured Information Extractor Using an Output Parser

**Stack:** Groq API + `openai/gpt-oss-120b` + Pydantic (schema enforcement / output parsing)

This notebook:
1. Installs/imports dependencies
2. Defines a strict Pydantic schema (`Candidate`)
3. Builds a prompt that forces the LLM to return JSON only, with `null` for missing fields
4. Calls the Groq API
5. Parses & validates the raw LLM output against the Pydantic schema (the "output parser")
6. Runs a test case with the provided candidate description


In [ ]:
# 1. Install dependencies
!pip install -q groq pydantic


In [ ]:
# 2. Imports & Groq client setup
import os
import json
from typing import List, Optional

from pydantic import BaseModel, ValidationError, field_validator
from groq import Groq

# Set your Groq API key (either as an env var beforehand, or paste it here)
# os.environ["GROQ_API_KEY"] = "your_api_key_here"

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "your_api_key_here")
MODEL_NAME = "openai/gpt-oss-120b"

client = Groq(api_key=GROQ_API_KEY)


In [ ]:
# 3. Pydantic schema (this IS our structured output parser)

class Candidate(BaseModel):
    Candidate_name: Optional[str] = None
    Years_of_experience: Optional[float] = None
    Current_role: Optional[str] = None
    Skills: Optional[List[str]] = None
    Highest_Education: Optional[str] = None

    @field_validator("Skills", mode="before")
    @classmethod
    def empty_list_is_fine(cls, v):
        # normalize missing/None skills to an empty list rather than breaking validation
        if v is None:
            return []
        return v

    class Config:
        extra = "forbid"  # reject any field not in the schema -> enforces strict schema


In [ ]:
# 4. Prompt template — instructs the model to output ONLY valid JSON,
#    following the exact schema, with null for any missing field.

SYSTEM_PROMPT = f"""You are a strict information extraction engine.

You will be given a free-text paragraph describing a candidate's background,
education, and experience (resume text, job application text, or self-introduction).

Extract the following fields and return ONLY a single valid JSON object,
with no markdown fences, no explanations, and no extra text before or after it.

JSON schema to follow EXACTLY (keys and types):
{{
  "Candidate_name": string or null,
  "Years_of_experience": number or null,
  "Current_role": string or null,
  "Skills": array of strings (use [] if none found),
  "Highest_Education": string or null
}}

Rules:
- If a field's value cannot be confidently found in the input text, return null for it
  (for Skills, return an empty array [] instead of null).
- Do not invent or guess information that is not present in the text.
- Years_of_experience must be a number (e.g. 3), not a string.
- Output must be valid JSON and nothing else.
"""

def build_messages(candidate_text: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": candidate_text},
    ]


In [ ]:
# 5. Extraction function: call Groq API -> parse JSON -> validate with Pydantic

def extract_candidate_info(candidate_text: str) -> Candidate:
    messages = build_messages(candidate_text)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=0,
        response_format={"type": "json_object"},  # force JSON-only output from Groq
    )

    raw_output = response.choices[0].message.content
    print("---- RAW MODEL OUTPUT ----")
    print(raw_output)
    print("--------------------------")

    # Step 1: parse raw text as JSON
    try:
        data = json.loads(raw_output)
    except json.JSONDecodeError as e:
        raise ValueError(f"Model did not return valid JSON: {e}")

    # Step 2: validate against the strict Pydantic schema (the output parser)
    try:
        candidate = Candidate(**data)
    except ValidationError as e:
        raise ValueError(f"Output failed schema validation:\n{e}")

    return candidate


In [ ]:
# 6. Test case — using the provided self-introduction text

test_text = (
    "My name is Bavly, I am an AI engineer with 3 years of experience, "
    "graduated from Suez University. Through this experience I built a strong "
    "foundation in ML, CV applications, AI agents, and now I am a trainee with "
    "Digital Hub as an AI engineer."
)

result = extract_candidate_info(test_text)

print("\n---- VALIDATED CANDIDATE OBJECT ----")
print(result)

print("\n---- FINAL JSON OUTPUT ----")
print(result.model_dump_json(indent=2))


In [ ]:
# 7. Extra test — text with missing fields, to confirm nulls are returned correctly

missing_fields_text = (
    "I love building things with computers and I'm passionate about technology."
)

result_missing = extract_candidate_info(missing_fields_text)

print("\n---- VALIDATED CANDIDATE OBJECT (missing fields case) ----")
print(result_missing)

print("\n---- FINAL JSON OUTPUT ----")
print(result_missing.model_dump_json(indent=2))

# Basic assertions to confirm the parser enforces the schema
assert isinstance(result_missing.Skills, list)
print("\nAll checks passed: schema enforced, missing fields handled as null/[] .")


---
# Part 2 — Same Task Using LangChain

This section re-implements the extractor using **LangChain** with:
- `ChatGroq` as the LLM wrapper (same `openai/gpt-oss-120b` model on Groq)
- `PydanticOutputParser` as the structured **output parser** (reusing the same `Candidate` schema)
- A `PromptTemplate` that injects the parser's format instructions automatically

Then we test it with text that **omits years of experience**, to confirm that field is correctly returned as `null`.


In [ ]:
# 8. Install LangChain + Groq integration
!pip install -q langchain langchain-groq langchain-core


In [ ]:
# 9. LangChain imports & setup

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

# Reuse the same Candidate Pydantic schema defined earlier

parser = PydanticOutputParser(pydantic_object=Candidate)

llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model=MODEL_NAME,
    temperature=0,
)

print(parser.get_format_instructions())


In [ ]:
# 10. Prompt template with format instructions baked in by the parser

lc_prompt = PromptTemplate(
    template=(
        "You are a strict information extraction engine.\n"
        "Extract candidate information from the text below.\n"
        "If a field is not explicitly present in the text, you MUST return null for it "
        "(use an empty array [] for Skills if none are found). Do not guess or invent values.\n\n"
        "{format_instructions}\n\n"
        "Candidate text:\n{candidate_text}\n"
    ),
    input_variables=["candidate_text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# Chain: prompt -> llm -> pydantic output parser
lc_chain = lc_prompt | llm | parser


In [ ]:
# 11. LangChain test — full candidate text (same as Part 1 test)

lc_result_full = lc_chain.invoke({"candidate_text": test_text})

print("---- LANGCHAIN RESULT (full info) ----")
print(lc_result_full)
print(lc_result_full.model_dump_json(indent=2))


In [ ]:
# 12. LangChain test — text with EXPERIENCE deliberately omitted,
#     to verify Years_of_experience correctly comes back as null

no_experience_text = (
    "My name is Bavly, I am an AI engineer, graduated from Suez University. "
    "I built a strong foundation in ML, CV applications, and AI agents, "
    "and now I am a trainee with Digital Hub as an AI engineer."
)

lc_result_no_exp = lc_chain.invoke({"candidate_text": no_experience_text})

print("---- LANGCHAIN RESULT (years_of_experience omitted from input) ----")
print(lc_result_no_exp)
print(lc_result_no_exp.model_dump_json(indent=2))

assert lc_result_no_exp.Years_of_experience is None, "Expected Years_of_experience to be null!"
print("\nPASSED: Years_of_experience correctly returned as null when not mentioned in text.")
